In [14]:
import json
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
CORPUS_PATH = Path("./data/corpus.jsonl")

docs = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line))

print(f"Documentos carregados: {len(docs)}")

Documentos carregados: 2000


In [16]:
texts = [(d["title"] + ". " + d["abstract"]) for d in docs]

In [17]:
# Lowercase, remoção de stopwords, limita aos 5000 termos mais relevantes e utiliza unigramas e bigramas
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

# "Aprende" o vocabulário e gera a matriz TF-IDF
x = vectorizer.fit_transform(texts)

print("Matriz TF-IDF:", x.shape)

Matriz TF-IDF: (2000, 5000)


In [1]:
# Função para aplicar o vectorizer.
# Utiliza similaridade do cosseno para calcular o score e ordena de forma decrescente
def search_knn(query: str, k: int = 100):
    q_vector = vectorizer.transform([query])
    scores = cosine_similarity(q_vector, x).ravel()
    top_idx = scores.argsort()[::-1][:k]

    results = []
    for rank, idx in enumerate(top_idx, 1):
        results.append((idx, float(scores[idx]), docs[idx]))

    return results

In [19]:
query = "Pre-training of text and layout for document image understanding"

# Recupera os 100 documentos mais próximos
results = search_knn(query, k=100)

for rank, (idx, score, doc) in enumerate(results, 1):
    print(f"{rank:2}. [{score:.4f}] {doc.get('title', '')}")
    print(f"{doc.get('arxiv_id')}")
    print()

 1. [0.5464] LayoutLLM: Large Language Model Instruction Tuning for Visually Rich Document Understanding
2403.14252

 2. [0.5423] Spatial Information Integration in Small Language Models for Document Layout Generation and Classification
2501.05497

 3. [0.3893] XY-Cut++: Advanced Layout Ordering via Hierarchical Mask Mechanism on a Novel Benchmark
2504.10258

 4. [0.3818] PP-DocBee: Improving Multimodal Document Understanding Through a Bag of Tricks
2503.04065

 5. [0.3499] HAND: Hierarchical Attention Network for Multi-Scale Handwritten Document Recognition and Layout Analysis
2412.18981

 6. [0.3406] DocSAM: Unified Document Image Segmentation via Query Decomposition and Heterogeneous Mixed Learning
2504.04085

 7. [0.3241] PELMS: Pre-training for Effective Low-Shot Multi-Document Summarization
2311.09836

 8. [0.2596] SAIL: Sample-Centric In-Context Learning for Document Information Extraction
2412.17092

 9. [0.2214] LDP: Generalizing to Multilingual Visual Information Extraction b

In [20]:
queries_path = Path("./eval/queries.tsv")  # qid<TAB>texto da query
runs_dir = Path("./runs");
runs_dir.mkdir(exist_ok=True)
run_path = runs_dir / "knn.trec"

queries = pd.read_csv(queries_path, sep="\t", names=["qid", "text"])

# Roda o KNN nas 10 queries e salva o top 5 resultados retornados
with open(run_path, "w", encoding="utf-8") as f:
    for _, q in queries.iterrows():
        for rank, (idx, score, d) in enumerate(search_knn(q["text"], k=5), 1):
            f.write(f"{q['qid']} Q0 {d['arxiv_id']} {rank} {score:.6f} knn_tfidf\n")

print("Run salva em:", run_path)

with open(run_path, "r", encoding="utf-8") as f:
    for i, line in zip(range(5), f):
        print(line.rstrip())

Run salva em: runs\knn.trec
q1 Q0 2403.14252 1 0.718804 knn_tfidf
q1 Q0 2503.04065 2 0.606443 knn_tfidf
q1 Q0 2504.04085 3 0.395459 knn_tfidf
q1 Q0 2311.09836 4 0.394544 knn_tfidf
q1 Q0 2501.05497 5 0.322519 knn_tfidf
